In [1]:
import os
import glob
import psycopg2
import pandas as pd
from psycopg2.extras import execute_values

# Database connection details
DB_NAME = "github_repos"
DB_USER = "postgres"
DB_PASSWORD = "Sphings@19"
DB_HOST = "localhost"
DB_PORT = "5432"

# Directory containing CSV files
RESULTS_FOLDER = "../results"

try:
    # Connect to PostgreSQL
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    cur = conn.cursor()

    # Get all CSV files in the results folder
    csv_files = glob.glob(os.path.join(RESULTS_FOLDER, "*.csv"))

    # Read and combine all CSV files into a single DataFrame
    all_data = pd.DataFrame()

    for file in csv_files:
        print(f"Reading {file}...")
        df = pd.read_csv(file)

        # Rename CSV columns to match the database
        df.rename(columns={
            'Hash': 'hash',
            'Project ID': 'project_id',
            'Version': 'version',
            'License': 'license',
            'Method Name': 'method_name',
            'File Location': 'file_location',
            'Function Code': 'function_code',
            'Repository URL': 'repository_url',
            'Query Project': 'query_project',
            'Violation': 'violation'
        }, inplace=True)

        all_data = pd.concat([all_data, df], ignore_index=True)

    # Remove duplicates based on (hash, project_id)
    all_data.drop_duplicates(subset=['hash', 'project_id'], inplace=True)

    # Generate unique ID by combining hash and project_id
    all_data['_id'] = all_data['hash'].astype(str) + "_" + all_data['project_id'].astype(str)

    # Convert DataFrame to a list of tuples for batch insert
    records_to_insert = [
        (
            row['_id'], row['hash'], row['project_id'], row['version'], row['license'], row['method_name'],
            row['file_location'], row['function_code'], row['repository_url'], row['query_project'], row['violation']
        ) for _, row in all_data.iterrows()
    ]

    # Insert all records in bulk
    insert_query = """
     INSERT INTO repository_data (
        _id, hash, project_id, version, license, method_name,
        file_location, function_code, repository_url, query_project, violation
    ) VALUES %s
    ON CONFLICT (hash, project_id, version) DO NOTHING;
    """
    
    
    execute_values(cur, insert_query, records_to_insert)

    # Commit changes
    conn.commit()
    print(f"Inserted {len(records_to_insert)} new records successfully.")

except Exception as e:
    print("Error:", e)

finally:
    # Close connection
    if conn:
        cur.close()
        conn.close()

Reading ../results/microsoft_photo-inspector_matches_2029676242.csv...
Reading ../results/microsoft_opencv_contrib_matches_1466417441.csv...
Reading ../results/microsoft_lis-test_matches_2231654531.csv...
Reading ../results/microsoft_nfc-ndef-tag-reader_matches_630188294.csv...
Reading ../results/microsoft_msphpsql_matches_488835669.csv...
Reading ../results/microsoft_music-explorer_matches_847561265.csv...
Reading ../results/microsoft_moto-trial-racer-wp_matches_153877410.csv...
Reading ../results/microsoft_camera-explorer_matches_89803343.csv...
Reading ../results/microsoft_WindowsAzureToolkitForEclipseWithJava_matches_245826013.csv...
Reading ../results/microsoft_protractor_matches_2599035066.csv...
Reading ../results/microsoft_matchem-poker-wp_matches_1255853605.csv...
Reading ../results/microsoft_drumkit-wp_matches_1742728251.csv...
Reading ../results/microsoft_TypeScriptSamples_matches_2534179007.csv...
Reading ../results/microsoft_simple-filter-mixer_matches_142382497.csv...
Rea